# Dataset folder structure
```
data/
│
├── images/                              # PNG files
│   ├── 0.png                           
│   │   .
│   │   .
│   │   .
│   └── 1.png                          
│
└── labels/                                # txt files
	├── 0.txt
	.
	.
	.
	└── 1.txt
```

In [ ]:
from pathlib import Path

MODEL='Kuzushiji'

EPOCHS=1

LR= 1e-4 #5e-6 

USE_CHECKPOINT = True

DATASET_PATH = Path('kaggle_kuzushiji')

DATASET_PATH.mkdir(parents=True, exist_ok=True)

PREPROCESS_DATA = False

AUGMENT_DATA = False

# Build lists of images and texts

In [ ]:
import csv
if PREPROCESS_DATA:
    with open(DATASET_PATH / 'unicode_translation.csv', mode='r') as table:
        reader = csv.reader(table)
        unicode_to_transcription = dict((rows[0], rows[1]) for rows in reader)
        unicode_to_transcription['U+770C'] = '県'
        unicode_to_transcription['U+4FA1'] = '価'
        unicode_to_transcription['U+7A83'] = '窃'
        unicode_to_transcription['U+515A'] = '党'
        unicode_to_transcription['U+5E81'] = '庁'
        unicode_to_transcription['U+5039'] = '倹'
        
    with open(DATASET_PATH / 'train.csv', mode='r') as table:
        reader = csv.reader(table)
        images_to_labels = dict((rows[0], rows[1]) for rows in reader)
        del images_to_labels['image_id']

In [ ]:
if PREPROCESS_DATA:
    data_dict = {}
    for id, label in images_to_labels.items():
        characters = label.split(' ')
        transcription = ''
        for i in range(len(characters)):
            if i % 5 == 0:
                transcription += unicode_to_transcription[characters[i]]
        data_dict['kaggle_kuzushiji/' + 'train_images/' + id + '.jpg'] = transcription
    print(list(data_dict.items())[0])

In [ ]:
import json
from util.augmentation import augment_kuzushiji_kaggle
from util.data_processing import save_labels, split_dataset

if PREPROCESS_DATA:
    save_labels(data_dict, DATASET_PATH / 'labels.json')

with open(str(DATASET_PATH / 'labels.json'), 'r') as fp:
    data_dict = json.load(fp)

(DATASET_PATH/'train_images'/'augmented').mkdir(parents=True, exist_ok=True)
if AUGMENT_DATA:
    augment_kuzushiji_kaggle(data_dict, 3, 3, 3)
    save_labels(data_dict, DATASET_PATH / 'augmented_labels.json')

In [ ]:
import json
from pathlib import Path

with open(str(DATASET_PATH / 'augmented_labels.json'), 'r') as fp:
    data_dict = json.load(fp)

print(f"{len(data_dict.items())} dict elements")

word_files = list(data_dict.items())
print(word_files[0])

In [ ]:
from util.data_processing import get_words_list


    
train_words = get_words_list(word_files)
print(f'Train size: {len(train_words)}')

# Build dataset and dataloader

In [ ]:
from util.data_processing import WORDSDataset
from dtrocr.config import DTrOCRConfig

config = DTrOCRConfig(max_position_embeddings=512
    # attn_implementation='flash_attention_2'
)
train_data = WORDSDataset(words=train_words, config=config)

In [ ]:
train_data[9]['labels'].shape

In [ ]:
import torch

mask = [False] * len(train_data.words)
i = 0
count = 0
for data in train_data:
    if data['labels'].shape != torch.Size([384]):
        count += 1
        print(i, count)
        mask[i] = True
    i += 1
        
print(count)

In [ ]:
train_data.words = [a for i,a in enumerate(train_data.words) if not mask[i]]
len(train_data.words)

In [ ]:
from torch.utils.data import DataLoader
import multiprocessing as mp
from util.data_processing import WordsSampler

i = 0

if USE_CHECKPOINT and Path(f'{MODEL} train_samp_ind.txt').exists():
    with open(f'{MODEL} train_samp_ind.txt', 'r') as f:
        i = int(f.readline())
        print(i)
sampler = WordsSampler(train_data, i = i)
print(sampler.seq[0])


    

train_dataloader = DataLoader(train_data, batch_size=32, sampler=sampler, shuffle=False, num_workers=mp.cpu_count())

# Model

In [ ]:
from util.model import get_model

model = get_model(config, MODEL)

# Training

In [ ]:
from util.model import train

train(model, train_dataloader, None,  EPOCHS, MODEL, LR, 'Kkanji')

In [ ]:
list1, list2 = [0], [0]
print(list1[-1], list2[-1])

# Test

In [ ]:
from dtrocr.model import DTrOCRLMHeadModel
from dtrocr.config import DTrOCRConfig
from dtrocr.processor import DTrOCRProcessor
import torch
torch.save(model.state_dict(), f'../models/{MODEL}.pt')
# model = DTrOCRLMHeadModel(DTrOCRConfig())
model.eval()
model.to('cpu')
test_processor = DTrOCRProcessor(DTrOCRConfig())

In [ ]:
from util.model import test

test(model, train_words, 10)